# RQ Cross-Country Statistical Comparison (Kruskal-Wallis, Mann-Whitney, OLS)

**Section 2.5 of `docs/paper_draft.md`.** Everything in Results (Section 3) so far is descriptive
(weighted pass rates, medians, percent growth) — no test has established whether the 9 countries'
broadband quality actually differs by more than sampling noise. This notebook builds that test,
**for both data sources, run separately.**

Objective — three tests, in order, run once per source:
1. **Kruskal-Wallis H-test**: do the countries' download-speed distributions differ at all?
2. **Pairwise Mann-Whitney U** (Holm-corrected): which specific country pairs differ?
3. **OLS** `mean_dl ~ log(GDP) + log(density) + C(country)`: does country still matter once
   GDP/density are controlled for? This is the term that can actually speak to the Singapore/Malaysia
   growth-outlier question raised in Discussion (Section 4).

Scope: **9 countries, both Ookla (fixed broadband) and NDT7 (broadband)** — Indonesia added
2026-07-30 as a genuine 9th country on both sources; Singapore's NDT7 data (also added 2026-07-30)
is usable here since this only needs province-level `mean_dl`/GDP/density, not the ISP-level `asn`
column it lacks.

**Sources are analyzed separately, never pooled.** NDT7 runs roughly 3x lower than Ookla in
absolute terms (MacMillan et al. 2023 — passive vs. active measurement), so a single Kruskal-Wallis
over pooled rows would return a source effect dressed up as country variance. The question this
notebook answers is "does the country ordering survive on both platforms independently," not
"is there one combined distribution."


In [1]:
# Setup
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

EXPORTS = Path('../../data/exports')
REFERENCE = Path('../../data/reference')


def join_key(s):
    """Normalised province key: strip diacritics, spaces, dashes, case.

    Needed because the NDT7 exports and the reference files disagree on province
    spelling. Vietnam is the worst case -- NDT7 strips spaces ('AnGiang') while the
    reference file does not ('An Giang'), giving *zero* exact matches on 63 provinces,
    plus 'Ba Ria-Vung Tau' (hyphen vs en-dash) and 'HoaBinh'/'Hoa Binh' (tone mark on a
    different vowel) and 'HoChiMinh' vs ASCII 'Ho Chi Minh'. Verified: this key matches
    698/698 Vietnam rows and 63/63 provinces with 0 collisions among the 64 reference
    provinces, so the aggressive normalisation is safe here.
    """
    import re, unicodedata as ud
    s = ud.normalize('NFD', str(s))
    s = ''.join(ch for ch in s if not ud.combining(ch))
    s = s.replace('\u0111', 'd').replace('\u0110', 'D')
    return re.sub(r'[^A-Za-z]', '', s).lower()
pd.set_option('display.width', 120)


## Load Ookla — 9 countries, fixed broadband

In [2]:
COUNTRY_FILES = {
    'Thailand':    'ookla_province_quarterly.csv',
    'Vietnam':     'ookla_vietnam_province_quarterly.csv',
    'Philippines': 'ookla_philippines_province_quarterly.csv',
    'Singapore':   'ookla_singapore_province_quarterly.csv',
    'Cambodia':    'ookla_cambodia_province_quarterly.csv',
    'Laos':        'ookla_laos_province_quarterly.csv',
    'Malaysia':    'ookla_malaysia_province_quarterly.csv',
    'Myanmar':     'ookla_myanmar_province_quarterly.csv',
    'Indonesia':   'ookla_indonesia_province_quarterly.csv',  # 9th country, added 2026-07-30
}

country_frames = []
for country, fname in COUNTRY_FILES.items():
    d = pd.read_csv(EXPORTS / fname)
    d['country'] = country
    country_frames.append(d)

sea_df = pd.concat(country_frames, ignore_index=True)
sea_df.rename(columns={'avg_d_mbps': 'mean_dl'}, inplace=True)
sea_df['source'] = 'Ookla'
sea_df.shape


(3222, 17)

In [3]:
# Reliability filter -- same is_reliable gate used everywhere else in this project (Section 2.2, Step 5)
ookla_reliable = sea_df[sea_df['is_reliable'] == True].copy()
ookla_reliable.groupby('country').size().rename('reliable_province_quarters')


country
Cambodia       300
Indonesia      408
Laos           216
Malaysia       192
Myanmar        150
Philippines    204
Singapore       60
Thailand       924
Vietnam        768
Name: reliable_province_quarters, dtype: int64

## Load NDT7 — 9 countries, broadband (pre-filtered reliable exports)

Uses the `*_reliable_province_quarterly.csv` exports directly (already `total_tests>=100` filtered
at prep time — NDT7 does not use the `n_tiles>=5` clause Ookla does; see `docs/paper_draft.md`
Contributions bullet on `is_reliable` for why). Singapore included (province-level fields only,
no ISP-level dependency here); Philippines/Malaysia's NDT7 data landed 2026-07-29 via the collaborator
ASN relabel documented in `HANDOFF.md`.


In [4]:
NDT7_COUNTRY_FILES = {
    'Thailand':    'ndt7_broadband_thailand_reliable_province_quarterly.csv',
    'Vietnam':     'ndt7_broadband_vietnam_reliable_province_quarterly.csv',
    'Philippines': 'ndt7_broadband_philippines_reliable_province_quarterly.csv',
    'Singapore':   'ndt7_broadband_singapore_reliable_province_quarterly.csv',
    'Cambodia':    'ndt7_broadband_cambodia_reliable_province_quarterly.csv',
    'Laos':        'ndt7_broadband_laos_reliable_province_quarterly.csv',
    'Malaysia':    'ndt7_broadband_malaysia_reliable_province_quarterly.csv',
    'Myanmar':     'ndt7_broadband_myanmar_reliable_province_quarterly.csv',
    'Indonesia':   'ndt7_broadband_indonesia_reliable_province_quarterly.csv',
}

REF_FILES = {'Vietnam': 'vietnam_reference.csv'}   # countries needing a reference re-join

ndt7_frames = []
for country, fname in NDT7_COUNTRY_FILES.items():
    d = pd.read_csv(EXPORTS / fname)
    d['country'] = country

    # Some NDT7 exports carry no GDP/density because the province-name join failed
    # upstream (see join_key). Without this repair those rows are silently dropped by
    # run_ols()'s dropna, which for Vietnam meant 698 rows -- the whole country -- and
    # produced a *fabricated* coefficient of 0.0 in the rank correlation below.
    need = ['gdp_per_capita_thb_2021', 'density_per_km2']
    if country in REF_FILES and d[need].notna().sum().sum() == 0:
        ref = pd.read_csv(REFERENCE / REF_FILES[country])
        ref['_k'] = ref['province_en'].map(join_key)
        ref = ref.drop_duplicates('_k').set_index('_k')[need]
        d['_k'] = d['province'].map(join_key)
        matched = d['_k'].isin(ref.index).sum()
        d = d.drop(columns=need).join(ref, on='_k').drop(columns='_k')
        print(f'  repaired {country} reference join: {matched}/{len(d)} rows matched')

    ndt7_frames.append(d)

ndt7_df = pd.concat(ndt7_frames, ignore_index=True)
ndt7_df.rename(columns={'avg_d_mbps': 'mean_dl'}, inplace=True)
ndt7_df['source'] = 'NDT7'

# Already reliable-filtered at export time, but re-apply for parity/robustness
ndt7_reliable = ndt7_df[ndt7_df['is_reliable'] == True].copy()
ndt7_reliable.groupby('country').size().rename('reliable_province_quarters')


  repaired Vietnam reference join: 698/698 rows matched


country
Cambodia        33
Indonesia      402
Laos            37
Malaysia       165
Myanmar         76
Philippines    204
Singapore       52
Thailand       872
Vietnam        698
Name: reliable_province_quarters, dtype: int64

## Tests 1-3, run once per source

Same three tests as before, wrapped in functions so they run identically on `ookla_reliable` and
`ndt7_reliable` without duplicating logic — and so results are reported side by side, never pooled.


In [5]:
def run_kruskal(df, label):
    groups = [g['mean_dl'].dropna().values for _, g in df.groupby('country')]
    country_order = list(df.groupby('country').groups.keys())
    h_stat, p_kw = stats.kruskal(*groups)
    print(f"[{label}] Kruskal-Wallis H = {h_stat:.2f}, p = {p_kw:.3e}, n groups = {len(groups)}, "
          f"n rows = {sum(len(g) for g in groups)}")
    print(f"[{label}]", "Reject H0 (countries differ)" if p_kw < 0.05 else "Fail to reject H0 (no detected difference)")
    return h_stat, p_kw, country_order


def run_pairwise(df, country_order, label):
    from itertools import combinations
    pairs = list(combinations(country_order, 2))
    raw_p, u_stats, n_a, n_b = [], [], [], []
    for c1, c2 in pairs:
        x = df.loc[df['country'] == c1, 'mean_dl'].dropna()
        y = df.loc[df['country'] == c2, 'mean_dl'].dropna()
        n_a.append(len(x)); n_b.append(len(y))
        u, p = stats.mannwhitneyu(x, y, alternative='two-sided')
        u_stats.append(u); raw_p.append(p)
    reject, p_holm, _, _ = multipletests(raw_p, alpha=0.05, method='holm')
    pdf = pd.DataFrame({
        'country_a': [p[0] for p in pairs], 'n_a': n_a,
        'country_b': [p[1] for p in pairs], 'n_b': n_b,
        'U': u_stats, 'p_raw': raw_p, 'p_holm': p_holm, 'significant': reject,
    }).sort_values('p_holm')
    n_sig = pdf['significant'].sum()
    print(f"[{label}] {n_sig} of {len(pdf)} country pairs differ significantly after Holm correction (alpha=0.05)")
    return pdf


def run_ols(df, label):
    model_df = df.dropna(subset=['mean_dl', 'gdp_per_capita_thb_2021', 'density_per_km2', 'country']).copy()
    model_df['log_gdp'] = np.log(model_df['gdp_per_capita_thb_2021'])
    model_df['log_density'] = np.log1p(model_df['density_per_km2'])
    formula = 'mean_dl ~ log_gdp + log_density + C(country)'
    # Province-clustered standard errors. Each province contributes 12 quarterly rows, so the
    # observations are not independent -- the uncorrected Durbin-Watson is 0.613, i.e. strong
    # positive autocorrelation, and naive SEs are therefore optimistic. Clustering by province
    # is the standard correction. Verified: every coefficient keeps its sign and significance;
    # SEs inflate by 0.83x-2.68x (log_gdp 1.30 -> 3.42, Singapore 6.30 -> 13.74).
    model_df = model_df.copy()
    model_df['_pid'] = model_df['country'].astype(str) + '|' + model_df['province'].astype(str)
    model = smf.ols(formula, data=model_df).fit(
        cov_type='cluster', cov_kwds={'groups': model_df['_pid']})
    # Record the countries actually fitted. Deriving the baseline by set-difference
    # against the *input* country list cannot distinguish "reference level, coef 0 by
    # construction" from "dropped entirely by dropna" -- that is exactly how Vietnam
    # acquired a fake 0.0 coefficient. Carry the truth forward instead.
    model.countries_ = sorted(model_df['country'].unique())
    dropped = sorted(set(df['country'].unique()) - set(model.countries_))
    print(f"[{label}] OLS: n = {len(model_df)}, R-sq = {model.rsquared:.3f}, "
          f"countries = {len(model.countries_)}")
    if dropped:
        print(f"[{label}] !! DROPPED (missing covariates, excluded from ranking): {dropped}")
    return model


## Ookla results

In [6]:
h_ookla, p_ookla, order_ookla = run_kruskal(ookla_reliable, 'Ookla')


[Ookla] Kruskal-Wallis H = 2810.07, p = 0.000e+00, n groups = 9, n rows = 3222
[Ookla] Reject H0 (countries differ)


In [7]:
pairwise_ookla = run_pairwise(ookla_reliable, order_ookla, 'Ookla')
pairwise_ookla


[Ookla] 35 of 36 country pairs differ significantly after Holm correction (alpha=0.05)

,country_a,n_a,country_b,n_b,U,p_raw,p_holm,significant
35,Thailand,924,Vietnam,768,680089.0,8.166546e-232,2.939956e-230,True
13,Indonesia,408,Thailand,924,0.0,1.587811e-186,5.557339e-185,True
14,Indonesia,408,Vietnam,768,95.0,1.711329e-175,5.818518e-174,True
6,Cambodia,300,Thailand,924,0.0,1.187198e-149,3.917752e-148,True
7,Cambodia,300,Vietnam,768,24.0,1.417381e-142,4.535618e-141,True
19,Laos,216,Thailand,924,0.0,3.907812e-116,1.211422e-114,True
20,Laos,216,Vietnam,768,390.0,7.412195e-111,2.223658e-109,True
31,Philippines,204,Thailand,924,621.0,1.660113e-109,4.814329e-108,True
9,Indonesia,408,Malaysia,192,0.0,4.968122e-87,1.391074e-85,True
28,Myanmar,150,Thailand,924,0.0,4.147331e-86,1.119779e-84,True


In [8]:
pairwise_ookla.loc[~pairwise_ookla['significant']]  # pairs that do NOT differ -- worth naming in Discussion


,country_a,n_a,country_b,n_b,U,p_raw,p_holm,significant
1,Cambodia,300,Laos,216,32009.0,0.815209,0.815209,False


In [9]:
ols_ookla = run_ols(ookla_reliable, 'Ookla')
print(ols_ookla.summary())


[Ookla] OLS: n = 3222, R-sq = 0.870, countries = 9
                            OLS Regression Results                            
Dep. Variable:                mean_dl   R-squared:                       0.870
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     913.1
Date:                Wed, 19 Aug 2026   Prob (F-statistic):          3.42e-201
Time:                        21:12:11   Log-Likelihood:                -16001.
No. Observations:                3222   AIC:                         3.202e+04
Df Residuals:                    3211   BIC:                         3.209e+04
Df Model:                          10                                         
Covariance Type:              cluster                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

In [10]:
# Country fixed-effect coefficients only, sorted -- this is what actually speaks to the
# Singapore/Malaysia growth-outlier question in Discussion (Section 4)
def clean_country_index(s):
    # patsy formats these as 'C(country)[T.Vietnam]' -- strip down to the plain country name
    return s.rename(index=lambda k: re.sub(r"^C\(country\)\[T\.(.+)\]$", r"\1", k))

import re
coefs = clean_country_index(ols_ookla.params.filter(like='C(country)')).sort_values(ascending=False)
pvals = clean_country_index(ols_ookla.pvalues.filter(like='C(country)'))
pd.DataFrame({'coef_vs_baseline': coefs, 'p_value': pvals.reindex(coefs.index)})


,coef_vs_baseline,p_value
Singapore,275.263195,2.580566e-89
Thailand,188.903776,0.000000e+00
Malaysia,106.600582,6.395018e-68
Vietnam,83.772659,1.954621e-275
Philippines,55.012453,4.160444e-37
Laos,2.918941,2.743214e-01
Myanmar,-9.834781,1.452220e-03
Indonesia,-18.844728,2.014235e-10


## NDT7 results

Same three tests, NDT7 broadband, 9 countries. **Caveat carried from `HANDOFF.md`/Table 1:** NDT7
sample sizes are far smaller and uneven across countries than Ookla's (Cambodia/Laos/Myanmar each
have well under 100 reliable province-quarters, vs. Thailand's 872) — treat any NDT7-only
significance call for those countries as provisional, and check `n_a`/`n_b` in the pairwise table
before reading a p-value.

**🚨 Data-quality note found while building this notebook (2026-07-30):** Vietnam's NDT7 export (`ndt7_broadband_vietnam_reliable_province_quarterly.csv`) has `gdp_per_capita_thb_2021` and `density_per_km2` entirely null (698/698 rows) — the province-reference merge in `vietnam_ndt7_prep.ipynb` is silently failing, most likely a province-name mismatch against `data/reference/vietnam_reference.csv`. This drops Vietnam from the NDT7 OLS model below entirely (not just as the reference level — see the printed baseline list), unlike Ookla where Vietnam has real GDP/density data. Not fixed here — this is a bug in a different notebook and needs its own look; flagging it rather than silently absorbing it into 'baseline'.


In [11]:
h_ndt7, p_ndt7, order_ndt7 = run_kruskal(ndt7_reliable, 'NDT7')


[NDT7] Kruskal-Wallis H = 1599.70, p = 0.000e+00, n groups = 9, n rows = 2539
[NDT7] Reject H0 (countries differ)


In [12]:
pairwise_ndt7 = run_pairwise(ndt7_reliable, order_ndt7, 'NDT7')
pairwise_ndt7


[NDT7] 35 of 36 country pairs differ significantly after Holm correction (alpha=0.05)


,country_a,n_a,country_b,n_b,U,p_raw,p_holm,significant
13,Indonesia,402,Thailand,872,858.0,1.231772e-179,4.434378e-178,True
14,Indonesia,402,Vietnam,698,20982.0,2.835438e-122,9.924032e-121,True
35,Thailand,872,Vietnam,698,493015.0,3.582292e-99,1.217979e-97,True
11,Indonesia,402,Philippines,204,1065.0,1.290410e-85,4.258354e-84,True
9,Indonesia,402,Malaysia,165,307.0,9.172151e-77,2.935088e-75,True
25,Malaysia,165,Vietnam,698,99410.0,8.502217e-48,2.635687e-46,True
28,Myanmar,76,Thailand,872,86.0,3.056051e-47,9.168153e-46,True
29,Myanmar,76,Vietnam,698,1401.0,5.809321e-42,1.684703e-40,True
26,Myanmar,76,Philippines,204,87.0,4.556526e-37,1.275827e-35,True
21,Malaysia,165,Myanmar,76,12518.0,1.950279e-35,5.265753e-34,True


In [13]:
pairwise_ndt7.loc[~pairwise_ndt7['significant']]


,country_a,n_a,country_b,n_b,U,p_raw,p_holm,significant
1,Cambodia,33,Laos,37,723.0,0.187599,0.187599,False


In [14]:
ols_ndt7 = run_ols(ndt7_reliable, 'NDT7')
print(ols_ndt7.summary())


[NDT7] OLS: n = 2539, R-sq = 0.647, countries = 9
                            OLS Regression Results                            
Dep. Variable:                mean_dl   R-squared:                       0.647
Model:                            OLS   Adj. R-squared:                  0.646
Method:                 Least Squares   F-statistic:                     134.0
Date:                Wed, 19 Aug 2026   Prob (F-statistic):           6.18e-91
Time:                        21:12:11   Log-Likelihood:                -11811.
No. Observations:                2539   AIC:                         2.364e+04
Df Residuals:                    2528   BIC:                         2.371e+04
Df Model:                          10                                         
Covariance Type:              cluster                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------

In [15]:
coefs_n = clean_country_index(ols_ndt7.params.filter(like='C(country)')).sort_values(ascending=False)
pvals_n = clean_country_index(ols_ndt7.pvalues.filter(like='C(country)'))
pd.DataFrame({'coef_vs_baseline': coefs_n, 'p_value': pvals_n.reindex(coefs_n.index)})


,coef_vs_baseline,p_value
Singapore,127.576412,1.138074e-10
Malaysia,56.262922,1.192159e-15
Thailand,46.490383,1.366292e-25
Philippines,28.891091,2.558198e-11
Vietnam,19.379949,4.570819e-07
Laos,2.417460,6.098815e-01
Myanmar,-4.567783,2.671191e-01
Indonesia,-12.365732,3.219806e-03


## Cross-source agreement check

Does the country ranking (by OLS fixed-effect coefficient) agree between the two sources, for the
countries both sources actually cover? This is the closest thing to a formal answer to "do Ookla and
NDT7 agree," beyond the descriptive Thailand/Vietnam comparison in `rq2_trends.ipynb`.


In [16]:
# C(country) fixed effects are relative to a baseline country (alphabetically first, dropped
# from the coefficient table) which gets coef=0 by construction -- add it back before ranking,
# otherwise it is silently excluded from the rank correlation.
#
# The baseline is derived from `.countries_`, i.e. the countries actually in the fitted model,
# NOT from the input country list. Using the input list conflates "reference level" with
# "dropped by dropna" and assigns a fabricated 0.0 to any country missing covariates.
baseline_ookla = sorted(set(ols_ookla.countries_) - set(coefs.index))
baseline_ndt7 = sorted(set(ols_ndt7.countries_) - set(coefs_n.index))
coefs_full = pd.concat([coefs, pd.Series(0.0, index=baseline_ookla)])
coefs_n_full = pd.concat([coefs_n, pd.Series(0.0, index=baseline_ndt7)])
print(f"Ookla baseline (coef=0 by construction): {baseline_ookla}")
print(f"NDT7 baseline (coef=0 by construction): {baseline_ndt7}")

common = sorted(set(order_ookla) & set(order_ndt7))
rank_df = pd.DataFrame({
    'Ookla_coef': coefs_full.reindex(common),
    'NDT7_coef':  coefs_n_full.reindex(common),
}).dropna()
rank_df['Ookla_rank'] = rank_df['Ookla_coef'].rank(ascending=False)
rank_df['NDT7_rank']  = rank_df['NDT7_coef'].rank(ascending=False)
spearman_r, spearman_p = stats.spearmanr(rank_df['Ookla_rank'], rank_df['NDT7_rank'])
print(f"Spearman rank correlation (country OLS coefficients, Ookla vs NDT7): rho={spearman_r:.2f}, p={spearman_p:.3f}, n={len(rank_df)} countries")
rank_df.sort_values('Ookla_rank')


Ookla baseline (coef=0 by construction): ['Cambodia']
NDT7 baseline (coef=0 by construction): ['Cambodia']
Spearman rank correlation (country OLS coefficients, Ookla vs NDT7): rho=0.97, p=0.000, n=9 countries


,Ookla_coef,NDT7_coef,Ookla_rank,NDT7_rank
Singapore,275.263195,127.576412,1.0,1.0
Thailand,188.903776,46.490383,2.0,3.0
Malaysia,106.600582,56.262922,3.0,2.0
Vietnam,83.772659,19.379949,4.0,5.0
Philippines,55.012453,28.891091,5.0,4.0
Laos,2.918941,2.417460,6.0,6.0
Cambodia,0.000000,0.000000,7.0,7.0
Myanmar,-9.834781,-4.567783,8.0,8.0
Indonesia,-18.844728,-12.365732,9.0,9.0


## Results summary

- **Kruskal-Wallis:** *(fill in H/p for both Ookla and NDT7 from the outputs above)*
- **Mann-Whitney pairs:** *(fill in how many pairs were significant for each source, and name any
  surprising non-significant pairs — check `n_a`/`n_b` for NDT7 pairs before trusting a call)*
- **OLS:** *(fill in R² for both sources, whether `C(country)` terms remain significant after
  GDP/density control, and whether Singapore/Malaysia's coefficients support or undercut the
  catch-up-growth story in Section 3.2 — on Ookla; note whether NDT7 agrees)*
- **Cross-source agreement:** *(fill in the Spearman rho from the cell above — does the country
  ranking hold up on both platforms, or does it fall apart outside Ookla)*

Once these are filled in, port the numbers into `docs/paper_draft.md` Section 2.5/3 (replace
"planned, not yet executed") and Section 5 Limitations (remove the "no test exists yet" line, and
update "NDT7 cross-validation covers only 2 of 8 countries" — it's now used across all 9 here,
albeit only for this cross-country stats test, not yet the RQ1/RQ2 narrative sections).


## Next steps

- Decide: continue, pivot, or stop, based on whether OLS country coefficients (either source)
  explain anything Section 4's Discussion couldn't already explain descriptively.
- If findings are strong, add a `docs/paper.tex`-ready results table (country coefficient, p-value,
  significance stars, per source) here before porting.
